# Module 04: A Memory the Agent Cannot Explain

Module 03 gave Sarah Chen's travel assistant a complete semantic-memory lifecycle. It can decide what is worth remembering, keep untrusted memories away from recommendations, replace outdated beliefs without deleting history, and retire low-value records.

Yet Sarah asks a simple question:

> **“Why do you think I prefer morning flights?”**

The graph contains more than the preference text. It also contains trust state, confidence, source, timestamps, and confirmation history. But does that information survive the trip from storage to the agent's response?

We will begin at the exact read boundary left by Module 03 and inspect what the agent actually receives.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import json
import os
import sys

import certifi
import sniffio

sys.path.insert(0, ".")
sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from azure.ai.projects.aio import AIProjectClient as AsyncAIProjectClient
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider,
)
from openai import AsyncAzureOpenAI
from agent_framework import Agent, AgentSession, tool
from lifecycle_utils import GraphProvenanceStore
from shared.travel_agent import (
    SYSTEM_PROMPT,
    create_client,
    get_travel_policy,
    search_flights,
    search_hotels,
)

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Foundry client ready")

In [ ]:
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.llm.adapters.openai import (
    OpenAIEmbeddingProvider,
    OpenAIProvider,
)
from pydantic import SecretStr

project_client = AsyncAIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=AsyncCliCredential(),
)
llm = OpenAIProvider(model=os.environ.get("FOUNDRY_MODEL", "gpt-4o"))
llm._client = project_client.get_openai_client()

embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(),
    "https://cognitiveservices.azure.com/.default",
)
embedder = OpenAIEmbeddingProvider(
    model=os.environ.get(
        "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
        "text-embedding-ada-002",
    )
)
embedder._client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token,
    api_version="2024-02-01",
)

settings = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USER"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
        "database": os.environ.get("NEO4J_DATABASE", "neo4j"),
    },
    llm=llm,
    embedding=embedder,
)
memory = MemoryClient(settings)
await memory.__aenter__()
provenance = GraphProvenanceStore(memory, user_id="E001")
await provenance.reset()
print("GraphProvenanceStore ready for Sarah Chen (E001)")

## The Problem Left at the End of Module 03

Module 3.3's gated recall returns `[KNOWN]` or `[LIKELY]`. Module 3.4's current recall returns the same broad distinction while filtering superseded values. Those are correct lifecycle decisions—but they are **lossy projections** of the graph.

To reproduce that boundary, we store two beliefs with different origins and evidence, then call `recall_baseline()`. The morning-flight inference is confirmed once only so it reaches `provisional` and becomes visible under the rules already established in Module 3.3.

In [ ]:
await provenance.store(
    category="hotel_chain",
    preference="Prefers Marriott hotels",
    source_type="user_assertion",
    confidence=0.95,
    source_detail="Sarah said: 'Marriott is always my first choice.'",
    created_by="Sarah Chen via TravelAssistant",
)
await provenance.store(
    category="flight_time",
    preference="May prefer morning flights",
    source_type="llm_inference",
    confidence=0.55,
    source_detail="Three recent booked trips departed before 9:00 AM.",
    created_by="TravelAssistant pattern inference",
)
await provenance.confirm("morning flights")  # candidate -> provisional

print(await provenance.recall_baseline("travel preferences"))

In [ ]:
@tool
async def recall_module3_style(query: str) -> str:
    """Recall current beliefs using only Module 03-style state labels."""
    return await provenance.recall_baseline(query)

baseline_agent = Agent(
    client=client,
    name="LifecycleOnlyTravelAssistant",
    instructions=SYSTEM_PROMPT + "\n\n" + (
        "Before describing the user's preferences, call recall_module3_style. "
        "Use only what that tool returns. If asked why a belief exists, do not "
        "invent a source or supporting evidence that the tool did not provide."
    ),
    tools=[recall_module3_style],
)

baseline_session = AgentSession()
result = await baseline_agent.run(
    "What travel preferences do you remember, and why do you think I prefer morning flights?",
    session=baseline_session,
)
print(result.text)

## What Went Wrong

The problem is not that Module 03 lost the data in Neo4j. The problem is that its read methods intentionally projected only the fields needed for their lesson.

| Module 03 capability | Question it answers | Question still unanswered |
|---|---|---|
| Identification | Should this be remembered? | Why was this particular memory created? |
| Staged promotion | May it influence the agent? | How strong was the supporting evidence? |
| Belief revision | Which value is current? | What caused each version to exist? |
| Retention | Is it worth keeping? | How should the agent communicate its certainty? |

Both visible records can be labelled `[LIKELY]`, even though one is Sarah's direct statement at `0.95` confidence and the other is an inference at `0.55`. The baseline output also contains no evidence with which to answer “why?”. Asking the model to fill that gap would produce a generated rationale, not an audit trail.

## The Missing Concept: Provenance

**Provenance is the traceable origin and evidence chain of a memory.** It should let the system answer:

1. **Who or what created it?** The user, an agent inference, a tool, or an enterprise source.
2. **Which interaction or evidence supported it?** The actual statement or observation—not merely a generic source label.
3. **When was it recorded and confirmed?** Timestamps place the evidence in context.
4. **How was it validated?** Trust state and confirmation count show what happened after creation.
5. **How did it change?** Lineage connects the current value to superseded versions.

`source_type="llm_inference"` alone is not enough. It identifies the kind of origin, but it does not explain the inference. `source_detail="Three recent booked trips departed before 9:00 AM"` provides inspectable evidence.

### Provenance is not confidence

- **Provenance** says *where and why* a memory came from.
- **Confidence** estimates *how strongly* the system should believe and communicate it.
- **Temporal history** says *how the value changed*.
- **Trust state** says *whether it may influence the agent now*.

Persisting provenance matters because a model can otherwise invent a convincing explanation after the fact. Stored evidence lets users inspect and correct memories, lets developers debug bad personalization, and makes the memory an auditable artifact rather than an unexplained assertion.

## The Solution: Preserve Metadata Through the Read Boundary

`GraphProvenanceStore` keeps Module 03's lifecycle rules, but returns structured provenance instead of reducing a node to text too early.

```mermaid
flowchart LR
    A[Neo4j Preference node] --> B{Lifecycle state}
    B -->|candidate / deprecated| C[Withhold]
    B -->|provisional / trusted| D[Structured recall]
    D --> E{State + confidence}
    E --> F[Assert]
    E --> G[Hedge]
    E --> H[Ask]
    A --> I[Source + evidence + actor + time]
    I --> J[Explain belief]
```

The order is important: **state controls visibility first; confidence controls wording second.**

| State | Confidence | Behavior |
|---|---:|---|
| `candidate` or `deprecated` | any | Withhold |
| `provisional` | below `0.5` | Ask |
| `provisional` | `0.5` or above | Hedge |
| `trusted` | `0.8` or above | Assert |
| `trusted` | `0.5`–`0.79` | Hedge |
| `trusted` | below `0.5` | Ask |

A high confidence score cannot bypass quarantine, and a provisional memory is never presented as settled fact.

In [ ]:
@tool
async def remember_belief(
    category: str,
    preference: str,
    source_type: str,
    confidence: float,
    source_detail: str,
) -> str:
    """Store a durable user belief and its provenance.

    source_type must be user_assertion, llm_inference, tool_output, or enterprise_kb.
    source_detail must quote or summarize the actual evidence that caused the memory.
    """
    result = await provenance.store(
        category=category,
        preference=preference,
        source_type=source_type,
        confidence=confidence,
        source_detail=source_detail,
        created_by="TravelAssistant",
    )
    return json.dumps(result)


@tool
async def confirm_belief(preference: str) -> str:
    """Record a later user reaffirmation of an existing belief."""
    return json.dumps(await provenance.confirm(preference))


@tool
async def recall_beliefs(query: str) -> str:
    """Recall current beliefs with mandatory assert, hedge, or ask behavior."""
    rows = await provenance.recall_with_confidence(query)
    if not rows:
        return "No current beliefs are eligible for recall."
    return json.dumps(rows, indent=2, default=str)


@tool
async def explain_belief(category: str) -> str:
    """Explain why the current belief in a category exists."""
    record = await provenance.explain(category)
    if record is None:
        return f"No current belief exists for category '{category}'."

    first_seen = record["first_seen"][:10] if record.get("first_seen") else "unknown"
    last_confirmed = (
        record["last_confirmed"][:10]
        if record.get("last_confirmed")
        else "not yet confirmed"
    )
    lineage = " -> ".join(item["preference"] for item in record["history"])
    evidence = record.get("source_detail") or "No supporting evidence was stored."
    return (
        f"Belief: {record['preference']}\n"
        f"Origin: {record['source_type']} via {record['created_by']}\n"
        f"Evidence: {evidence}\n"
        f"First recorded: {first_seen}\n"
        f"Confidence: {record['confidence']:.2f}\n"
        f"State: {record['state']} ({record['confirmation_count']} confirmations; "
        f"last confirmed: {last_confirmed})\n"
        f"Lineage: {lineage}"
    )


print("Provenance-aware tools ready")

In [ ]:
assistant = Agent(
    client=client,
    name="ProvenanceAwareTravelAssistant",
    instructions=SYSTEM_PROMPT + "\n\n" + (
        "You manage Sarah Chen's long-term travel preferences.\n"
        "- When Sarah states a durable preference, call remember_belief with "
        "source_type='user_assertion', an appropriate confidence, and source_detail "
        "that faithfully quotes or summarizes her statement.\n"
        "- When you infer a durable preference from observations, call remember_belief "
        "with source_type='llm_inference' and list the actual observations in source_detail.\n"
        "- If Sarah reaffirms a previously stored belief, call confirm_belief instead of "
        "storing a duplicate.\n"
        "- Before personalizing a response, call recall_beliefs. Obey every presentation "
        "field exactly: assert as fact, hedge as uncertain and seek confirmation, or ask "
        "a direct clarification question. Never use a belief absent from recall.\n"
        "- For questions containing 'why', 'how do you know', or 'where did that come "
        "from', call explain_belief. Use only its stored evidence; never invent provenance."
    ),
    tools=[
        search_flights,
        search_hotels,
        get_travel_policy,
        remember_belief,
        confirm_belief,
        recall_beliefs,
        explain_belief,
    ],
)
print(f"Agent ready: {assistant.name}")

## The Payoff: The Same Lifecycle, Now Explainable

Reset only Module 04's records and replay the scenario through the improved agent. The lifecycle behavior does not change: a fresh user assertion is provisional, and an inference begins as a hidden candidate. What changes is that recall retains enough metadata to choose honest language and explain the memory later.

In [ ]:
await provenance.reset()
session = AgentSession()

r1 = await assistant.run(
    "Marriott is always my first choice for business travel. Please remember that.",
    session=session,
)
print("ASSISTANT:", r1.text, "\n")
print(json.dumps(await provenance.snapshot(), indent=2, default=str))

# The direct statement is high confidence, but its provisional state still requires hedging.
r1_recall = await assistant.run(
    "Which hotel chain should you prioritize for me?",
    session=session,
)
print("\nASSISTANT:", r1_recall.text)

In [ ]:
# A later reaffirmation promotes the user assertion from provisional to trusted.
r2 = await assistant.run(
    "Yes, Marriott really is my preferred chain—please treat that as confirmed.",
    session=session,
)
print("ASSISTANT:", r2.text, "\n")
print(json.dumps(await provenance.recall_with_confidence("hotel preference"), indent=2, default=str))

r2_recall = await assistant.run(
    "Now which hotel chain should you prioritize?",
    session=session,
)
print("\nASSISTANT:", r2_recall.text)

In [ ]:
# An observed pattern is an inference, not a user assertion. It starts as a hidden candidate.
r3 = await assistant.run(
    "Review this pattern and remember any durable preference you infer: my last three "
    "booked trips departed at 7:10 AM, 8:00 AM, and 8:35 AM.",
    session=session,
)
print("ASSISTANT:", r3.text, "\n")
print("Stored records:")
print(json.dumps(await provenance.snapshot(), indent=2, default=str))
print("\nEligible recall (the candidate inference should be absent):")
print(json.dumps(await provenance.recall_with_confidence("flight time"), indent=2, default=str))

In [ ]:
# One confirmation makes an inferred candidate provisional—not trusted.
r4 = await assistant.run(
    "That pattern sounds right; I often prefer morning departures.",
    session=session,
)
print("ASSISTANT:", r4.text, "\n")
print(json.dumps(await provenance.recall_with_confidence("flight time"), indent=2, default=str))

r4_recall = await assistant.run(
    "What flight time should you prioritize for my next trip?",
    session=session,
)
print("\nASSISTANT:", r4_recall.text)

## Low Confidence Means Ask, Not Guess

A visible memory is not automatically safe to state. The following deliberately ambiguous user statement is stored as `provisional` with low confidence. This small fixture isolates the presentation policy: recall should return `ask`, prompting clarification rather than personalization.

In [ ]:
await provenance.store(
    category="seat",
    preference="Might prefer aisle seats",
    source_type="user_assertion",
    confidence=0.35,
    source_detail="Sarah said: 'I might prefer aisle seats, but I am not sure yet.'",
    created_by="Sarah Chen via TravelAssistant",
)
print(json.dumps(await provenance.recall_with_confidence("seat preference"), indent=2, default=str))

r5 = await assistant.run(
    "Use what you remember to choose my seat.",
    session=session,
)
print("\nASSISTANT:", r5.text)

## Explain the Belief, Not a Generated Story

The explanation tool reads persisted fields. A direct statement and an inferred pattern therefore produce different, inspectable explanations. If evidence was not stored, the tool says so instead of asking the model to reconstruct a plausible reason.

In [ ]:
r6 = await assistant.run(
    "Why do you believe I prefer Marriott?",
    session=session,
)
print("ASSISTANT:", r6.text)

r7 = await assistant.run(
    "How do you know I may prefer morning flights?",
    session=session,
)
print("\nASSISTANT:", r7.text)

print("\nSTRUCTURED PROVENANCE (flight_time):")
print(json.dumps(await provenance.explain("flight_time"), indent=2, default=str))

## Provenance Across a Revision

Module 3.4 already taught why old values are retired rather than overwritten. Here we reuse that lineage for a different question: **what evidence caused each version to exist?** The temporal mechanism is unchanged; provenance makes each point in the lineage explainable.

In [ ]:
r8 = await assistant.run(
    "My company changed its preferred vendor. Hilton is now my first choice instead of Marriott.",
    session=session,
)
print("ASSISTANT:", r8.text, "\n")

hotel_explanation = await provenance.explain("hotel_chain")
print(json.dumps(hotel_explanation, indent=2, default=str))

r9 = await assistant.run(
    "Why do you now think Hilton is my first choice, and what did you believe before?",
    session=session,
)
print("\nASSISTANT:", r9.text)

## Key Takeaways

1. **Module 03 was necessary but not sufficient.** It made memories eligible, current, and bounded; Module 04 preserves the reasons behind them through recall.
2. **Provenance is an evidence chain, not a source label.** Store the actor, source type, supporting interaction or observation, timestamps, confirmations, and lineage.
3. **State comes before confidence.** Candidates remain hidden regardless of score; provisional beliefs never become facts merely because their numeric confidence is high.
4. **Confidence changes language.** The same memory can require an assertion, a hedge, or a clarification question.
5. **Explanations must be retrieved, not improvised.** If evidence was not persisted, the agent must say it is unavailable.
6. **One graph remains the source of truth.** Provenance is metadata on the existing preference nodes, while `GraphProvenanceStore` keeps this lesson's Python interface focused.

## Next: Module 05 — Retrieval and Routing

Now that recalled memories retain trust, confidence, and provenance, the next module decides which memory system should answer a request and how the most relevant records reach the agent.

In [ ]:
# Cleanup for a repeatable demo. Run this after inspecting the records above.
await provenance.reset()
await memory.__aexit__(None, None, None)
await project_client.close()
await credential.close()
print("Module 04 demo records removed and clients closed")